# RapidEye NITF with RPC Geolocation**Source**: `grdl/example/IO/eo/rapideye_nitf_rpc.py`## Learning Objectives- Load EO NITF files with RPC metadata- Build RPCGeolocation from RPC00B TRE- Validate RPC projection accuracy## Data Setup**Path Configuration**: Set path to a RapidEye (or similar) NITF file with RPC.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
# ConfigurationNITF_PATH = '/path/to/your/rapideye.ntf'CHIP_SIZE = 1024print(f"NITF file: {NITF_PATH}")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom grdl.IO import get_readerfrom grdl.geolocation.eo.rpc import RPCGeolocation

In [ ]:
# Load NITF and extract RPC metadatawith get_reader('eo-nitf', NITF_PATH) as reader:    metadata = reader.metadata    rpc = metadata.rpc_coefficients if hasattr(metadata, 'rpc_coefficients') else None        if rpc is None:        raise ValueError("No RPC metadata found in NITF file")        print(f"\nRPC coefficients found:")    print(f"  Error bias (row, col): ({rpc.error_bias_row}, {rpc.error_bias_col})")    print(f"  Error random (row, col): ({rpc.error_rand_row}, {rpc.error_rand_col})")        # Build RPC geolocation    rows, cols = reader.get_shape()    geo = RPCGeolocation(rpc, shape=(rows, cols))        # Test corner projection    corners = [(0, 0), (0, cols-1), (rows-1, 0), (rows-1, cols-1)]    print(f"\nCorner coordinates:")    for r, c in corners:        lat, lon, _ = geo.image_to_latlon(r, c)        print(f"  Pixel ({r:5d}, {c:5d}) → ({lat:.6f}°, {lon:.6f}°)")

In [ ]:
# Load chip for displaywith get_reader('eo-nitf', NITF_PATH) as reader:    r0 = max(0, rows // 2 - CHIP_SIZE // 2)    c0 = max(0, cols // 2 - CHIP_SIZE // 2)    chip = reader.read_chip(r0, r0 + CHIP_SIZE, c0, c0 + CHIP_SIZE)        print(f"\nChip extracted: [{r0}:{r0+CHIP_SIZE}, {c0}:{c0+CHIP_SIZE}]")    print(f"Chip shape: {chip.shape}")

In [ ]:
# Visualizefig, ax = plt.subplots(figsize=(10, 10))if chip.ndim == 3 and chip.shape[2] >= 3:    # RGB composite    ax.imshow(chip[:, :, :3])else:    # Grayscale    ax.imshow(chip, cmap='gray')ax.set_title(f'RapidEye NITF with RPC Geolocation', fontsize=14)plt.tight_layout()plt.show()

## Summary**GRDL patterns**:- ✅ `get_reader('eo-nitf', path)` — EO NITF reader- ✅ `RPCGeolocation` — RPC00B rational polynomial geolocation- ✅ Automatic RPC extraction from NITF TREs